# 03 - Reproducible Splitting

Creates the **one** frozen stratified train/validation/test split (random seed 42) used by
every method in this project. Saves `train.csv`, `validation.csv`, `test.csv`,
`split_manifest.json`, the dataset fingerprint, and the class distribution for every split.

The same untouched `test.csv` is the only test set ever used to evaluate BERT, the LLM, and
LLM+RAG. Test documents must never appear in BERT training/validation, prompt-development
examples, threshold tuning, or the RAG index.

All logic lives in `src/newstart_ai/data/splitting.py` and `fingerprinting.py`. Outputs go to
`data/splits/`.

### Load configuration and the full dataset

**Purpose:** Load the same fixed dataset used in notebooks 01 and 02, plus the splitting
functions this notebook will use to create the project's one and only train/validation/test
split.

**Why this step is necessary:** Splitting must start from the *full, validated* dataset --
this is the last notebook that ever touches all 754 rows together. From this point on,
every other notebook works with one specific split file (`train.csv`, `validation.csv`, or
`test.csv`) and never recombines them.

**Inputs:** `configs/base.yaml` (via `settings`) and `data/processed/final_dataset.csv`.

**Output:** `settings` and `df` (the full 754-row dataset).

**How to interpret the result:** Just a row-count confirmation; the actual split happens in
the next cell.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import create_stratified_split, load_dataset, save_split

settings = load_settings()
df = load_dataset(settings)
print(f"{len(df)} documents loaded from {settings.base.dataset.path}")

754 documents loaded from data/processed/final_dataset.csv


## Create the split

Uses `configs/base.yaml: split` (train 0.64 / validation 0.16 / test 0.20, seed 42). Running
this cell again with the same dataset and config reproduces the identical split -- the seed
is fixed and stratification is deterministic given the same input order handling in
scikit-learn.

### Create the one frozen stratified split

**Purpose:** Split the full dataset into training (64%), validation (16%), and test (20%)
subsets, stratified so each split has roughly the same class proportions as the full
dataset, using a fixed random seed for reproducibility.

**Why this step is necessary:** This is the single most important reproducibility
guarantee in the whole project: every method (BERT, LLM, LLM+RAG) is compared on the exact
same test documents, and none of them are allowed to train, tune, or retrieve from those
documents beforehand. A fixed seed means re-running this cell later reproduces the same
split rather than a new random one. Stratifying (rather than a plain random split) matters
because IRS is such a small class -- an unstratified split could easily leave IRS
completely absent from one of the three subsets by chance.

**Inputs:** `df` (the full dataset) and the split ratios/seed from
`configs/base.yaml: split`.

**Output:** `train_df`, `val_df`, `test_df` (three DataFrames) and `manifest`, a
`SplitManifest` object recording exactly which document IDs went where, plus a fingerprint
of the source dataset's content.

**How to interpret the result:** The three row counts should roughly match a 64/16/20 split
of 754 documents. The fingerprint is a content hash -- if `final_dataset.csv` ever changes,
this fingerprint would change too, which is how a future run could detect that a saved split
no longer matches the current dataset.

In [2]:
train_df, val_df, test_df, manifest = create_stratified_split(df, settings)
print(f"train: {len(train_df)}  validation: {len(val_df)}  test: {len(test_df)}")
print(f"dataset fingerprint: {manifest.dataset_fingerprint}")

train: 482  validation: 121  test: 151
dataset fingerprint: dcbdbbcb287b203c1faae8a4b3583980d92149e116d3ad87f86cb5bab4ced3f3


## Leakage check

Proves no document_id appears in more than one split -- this is the concrete guarantee every downstream notebook (BERT, RAG index, prompt development) relies on.

### Prove there is no leakage between splits

**Purpose:** Explicitly check that no single document ID appears in more than one of
`train_df`, `val_df`, and `test_df`.

**Why this step is necessary:** This is not a theoretical concern -- it's the concrete,
checkable guarantee that every later notebook depends on. If a test document leaked into
training, BERT could partially "memorize" it, and its test-set performance would no longer
be a fair, honest measurement. Checking this immediately after creating the split, rather
than assuming the splitting code is correct, is the whole point of this cell.

**Inputs:** `manifest` (specifically, the three lists of document IDs it records).

**Output:** Either an exception (if leakage were found) or a printed confirmation.

**How to interpret the result:** Seeing the confirmation message is what allows every later
notebook to trust that `test.csv` is truly untouched by anything upstream of it.

In [3]:
manifest.assert_no_overlap()
print("No overlap between train/validation/test document IDs.")

No overlap between train/validation/test document IDs.


## Class distribution per split

Confirms the split is stratified and shows exactly how few IRS documents land in each split -- this is the numeric basis for the IRS test-set-size caveat used throughout the rest of the project.

### Tabulate class distribution per split

**Purpose:** Show exactly how many documents of each agency ended up in each of the three
splits.

**Why this step is necessary:** "Stratified" is a claim about the splitting algorithm's
*intent* -- this table is the evidence that it actually worked, and more importantly, it's
where the small IRS test-set size becomes a concrete, visible number rather than an
abstract worry.

**Inputs:** `manifest.class_distribution`, computed as part of creating the split above.

**Output:** A pivot table with one row per agency and one column per split.

**How to interpret the result:** Each agency should appear in all three splits in roughly
64/16/20 proportions. IRS, being the smallest class overall, will have the smallest raw
counts in every split -- most importantly, only a handful of IRS documents in `test.csv`
(quantified in the next cell).

In [4]:
import pandas as pd

dist_df = pd.DataFrame([cd.model_dump() for cd in manifest.class_distribution])
dist_df.pivot(index="label", columns="split", values="count").fillna(0).astype(int)

split,test,train,validation
label,,,
DMV,55,178,44
IRS,5,14,4
SSA,40,126,32
USCIS,51,164,41


### Call out the IRS test-set size explicitly

**Purpose:** Pull the exact IRS row count in the test split out of the table above and
state, in plain language, what that small number means for how IRS results should be
interpreted later.

**Why this step is necessary:** A small test-set size for one class doesn't invalidate the
split (IRS is simply a small class in the source dataset), but it does mean IRS's precision,
recall, and F1 in every later evaluation notebook will be based on only a handful of
examples and can swing dramatically from a single misclassification. Stating this plainly,
right where the split is created, means every later notebook can refer back to this as the
documented reason for treating IRS metrics cautiously.

**Inputs:** `dist_df` (computed above), filtered to the IRS row of the test split.

**Output:** Printed text.

**How to interpret the result:** Expect roughly 4-5 IRS documents in the test split (about
20% of IRS's ~23 total documents). Any IRS precision/recall/F1 number reported in notebooks
05, 06, 08, 09, or 10 should be read with this small denominator in mind.

In [5]:
irs_test_count = dist_df.query("label == 'IRS' and split == 'test'")["count"].iloc[0]
print(
    f"IRS test-set size: {irs_test_count} documents. "
    "Per docs/BLUEPRINT.md Section 6, IRS per-class metrics from this split must always be "
    "reported with an explicit small-sample caveat, not as precise results."
)

IRS test-set size: 5 documents. Per docs/BLUEPRINT.md Section 6, IRS per-class metrics from this split must always be reported with an explicit small-sample caveat, not as precise results.


## Save the split

Writes `train.csv`, `validation.csv`, `test.csv`, and `split_manifest.json` to `data/splits/`. From this point on, every other notebook only *reads* these files -- the split is never recomputed.

### Save the split to disk

**Purpose:** Write `train.csv`, `validation.csv`, `test.csv`, and `split_manifest.json` to
`data/splits/`, making this split a persistent, shareable artifact rather than something
that only exists in this notebook's memory.

**Why this step is necessary:** Every later notebook (04 onward) loads these exact files
from disk via `load_split()` -- they never re-run `create_stratified_split()` themselves.
Saving here, once, is what makes the split genuinely "frozen": as long as these files exist,
re-running this notebook again would only overwrite them with an identical result (same
seed, same source data), not silently produce a different split.

**Inputs:** `train_df`, `val_df`, `test_df`, and `manifest`.

**Output:** Four files written under `data/splits/`, and their filenames printed as
confirmation.

**How to interpret the result:** As long as these four files are present, every downstream
notebook can assume the split is fixed and load it directly -- there's no need to (and no
notebook should) call `create_stratified_split()` again after this point.

In [6]:
output_dir = save_split(train_df, val_df, test_df, manifest, settings)
print(f"Saved split files to {output_dir}")
for f in sorted(output_dir.iterdir()):
    print(" -", f.name)

Saved split files to D:\USD\Projects\a590\newstart-ai\newstart_ai_benchmark\data\splits


 - split_manifest.json
 - test.csv
 - train.csv
 - validation.csv


## Summary for the next notebooks

- The split is frozen: seed 42, 64/16/20 train/validation/test.
- `test.csv` is untouched from this point forward until each method's single frozen
  evaluation notebook.
- IRS's small test slice is a known, documented limitation, not an error.
- `04_bert_fine_tuning.ipynb` loads `train.csv` and `validation.csv` only.